In [0]:
# ════════════════════════════════════════════════════════════
# US GOVERNMENT HOLIDAY CALENDAR PIPELINE
# ════════════════════════════════════════════════════════════
#
# PURPOSE:
#   Automatically generate US Federal Holiday Calendar
#   and store it permanently in Databricks Delta table.
#
# SCHEDULE:
#   Runs automatically every January 1 at 12:00 AM
#   via Databricks Job
#
# OUTPUT TABLE:
#   workspace.default.us_holiday_calendar
#
# AUTHOR : Paresh Ranjan Rout
# EMAIL  : paresh.r@datapoem.com
# DATE   : April 2026
# VERSION: 2.0 — Incremental load
#
# ════════════════════════════════════════════════════════════

print("=" * 55)
print("  US Government Holiday Calendar Pipeline v2.0")
print("=" * 55)

  US Government Holiday Calendar Pipeline v2.0


In [0]:
# ════════════════════════════════════════════════════════════
# MAIN PIPELINE — INCREMENTAL LOAD
# ════════════════════════════════════════════════════════════
#
# LOGIC:
#   1. Check if current year data already exists in table
#      → YES : skip — do nothing — exit cleanly
#      → NO  : generate current year data only
#   2. If table exists → INSERT INTO (append only)
#      If table missing → CREATE TABLE (first time only)
#
# WHY INCREMENTAL?
#   → History is already loaded — no need to reprocess
#   → Only NEW year data is generated each run
#   → INSERT INTO never deletes old data — always safe!
#   → If pipeline re-runs manually — smart skip prevents duplicates
#
# ════════════════════════════════════════════════════════════

# ── IMPORTS ──────────────────────────────────────────────────
from datetime import datetime
from pandas.tseries.holiday import USFederalHolidayCalendar
import pandas as pd

# ── CONFIGURATION ────────────────────────────────────────────
# Table where holiday data is stored permanently
TABLE_NAME   = "workspace.default.us_holiday_calendar"

# Current year — calculated automatically!
# In 2027 this will be 2027, in 2028 this will be 2028 — auto!
CURRENT_YEAR = datetime.now().year

# Start and end date of current year
START_DATE   = datetime(CURRENT_YEAR, 1, 1)
END_DATE     = datetime(CURRENT_YEAR, 12, 31)

print(f"Pipeline year  : {CURRENT_YEAR}")
print(f"Date range     : {START_DATE.date()} to {END_DATE.date()}")
print(f"Target table   : {TABLE_NAME}")
print("-" * 55)


# ── STEP 1: CHECK IF CURRENT YEAR ALREADY EXISTS ─────────────
# Before doing any work — check if data is already in the table
# Prevents duplicate data and unnecessary compute cost
print(f"\nStep 1 — Checking if {CURRENT_YEAR} data exists...")

try:
    # Count rows for current year in the table
    row_count = spark.sql(f"""
        SELECT COUNT(*) as cnt
        FROM {TABLE_NAME}
        WHERE year = {CURRENT_YEAR}
    """).collect()[0][0]

    if row_count > 0:
        # Data already exists — exit cleanly — nothing to do!
        print(f"  ✓ {CURRENT_YEAR} already has {row_count} rows in table!")
        print(f"  ✓ Skipping — no action needed!")
        dbutils.notebook.exit(
            f"Pipeline skipped — {CURRENT_YEAR} data already present ({row_count} rows)."
        )
    else:
        # Table exists but current year is missing — process it!
        print(f"  {CURRENT_YEAR} not found — will generate and append!")
        table_exists = True

except Exception:
    # Table does not exist yet — very first run!
    print(f"  Table not found — first time setup — will create!")
    table_exists = False


# ── STEP 2: GENERATE CURRENT YEAR DATA ───────────────────────
# Only generates data for CURRENT YEAR — not history!
# History is already safe in the table — never reprocessed!
print(f"\nStep 2 — Generating {CURRENT_YEAR} calendar data...")

# Get US Federal Holidays for current year from Python library
# No API, no internet — rules are built into Python already!
cal      = USFederalHolidayCalendar()
holidays = cal.holidays(start=START_DATE, end=END_DATE)

# Build DataFrame — one row per day of the year
df = pd.DataFrame()
df['date']           = pd.date_range(start=START_DATE, end=END_DATE)
df['year']           = df['date'].dt.year.astype(int)
df['month']          = df['date'].dt.month.astype(int)
df['day']            = df['date'].dt.day.astype(int)
df['day_name']       = df['date'].dt.day_name()
df['week_number']    = df['date'].dt.isocalendar().week.astype(int)
df['is_holiday']     = [1 if d in holidays else 0 for d in df['date']]
df['is_weekend']     = df['date'].dt.dayofweek.isin([5, 6]).astype(int)
df['is_working_day'] = ((df['is_holiday'] == 0) & (df['is_weekend'] == 0)).astype(int)
df['date']           = df['date'].astype(str)

print(f"  ✓ Generated {len(df)} rows for {CURRENT_YEAR}")
print(f"  Holidays    : {int(df['is_holiday'].sum())}")
print(f"  Weekends    : {int(df['is_weekend'].sum())}")
print(f"  Working days: {int(df['is_working_day'].sum())}")


# ── STEP 3: SAVE TO DELTA TABLE ──────────────────────────────
# Append new year to existing table — NEVER replace!
# INSERT INTO = adds rows only — history always safe!
print(f"\nStep 3 — Saving {CURRENT_YEAR} data to Delta table...")

# Convert to Spark DataFrame for Delta write
spark.createDataFrame(df).createOrReplaceTempView("temp_current_year")

if table_exists:
    # Table exists — APPEND current year only!
    spark.sql(f"""
        INSERT INTO {TABLE_NAME}
        SELECT * FROM temp_current_year
    """)
    print(f"  ✓ {CURRENT_YEAR} data appended to {TABLE_NAME}")
    print(f"  ✓ All historical data is safe and untouched!")

else:
    # First time — table does not exist — CREATE it!
    spark.sql(f"""
        CREATE TABLE {TABLE_NAME}
        AS SELECT * FROM temp_current_year
    """)
    print(f"  ✓ Table created: {TABLE_NAME}")
    print(f"  Note: Load historical data 2020-{CURRENT_YEAR-1} separately!")


# ── STEP 4: VERIFY ───────────────────────────────────────────
# Read back from table and confirm data is correct
print(f"\nStep 4 — Verifying table...")

spark.sql(f"""
    SELECT
        year,
        COUNT(*)            AS total_days,
        SUM(is_holiday)     AS holidays,
        SUM(is_weekend)     AS weekends,
        SUM(is_working_day) AS working_days
    FROM {TABLE_NAME}
    GROUP BY year
    ORDER BY year
""").show()

print("=" * 55)
print(f"  ✓ Pipeline complete!")
print(f"  ✓ Next run: January 1, {CURRENT_YEAR + 1} at 12:00 AM")
print("=" * 55)